# Import Modules

In [1]:
import numpy as np
from mESPIRA import rESPIRA,FC_arr,iESPIRA
from itertools import permutations

# Define Error Calculation

In [19]:
# Computes the Approximation errors
def compare(w1,f1,w2,f2):
    #### Need to sort frequencies first
    def cartesian_product(*arrays):
        la = len(arrays)
        dtype = np.result_type(*arrays)
        arr = np.empty([len(a) for a in arrays] + [la], dtype=dtype)
        for i, a in enumerate(np.ix_(*arrays)):
            arr[...,i] = a
        return arr.reshape(-1, la)
    
    d = f1.shape[1]
    
    I = list(set(permutations(np.arange(len(f1)))))
    R = []
    for i in I:
        R+= [np.max(abs(f1-f2[np.array(i)]))]
    I = np.array(I[np.argmin(R)])

    f2 = f2[I]
    w2 = w2[I]
    #### End sorting
    
    try:
        print('err(freq)       \t: ',max([max(abs((f1-f2).T[d])/max(abs(f1).T[d])) for d in range(d)]))
        print('err(weig)       \t: ',np.max(abs(w1-w2))/np.max(abs(w1)))
    except:
        print('Failure to find correct frequencies / weights.')
    
    
    SOE1 = lambda x : w1@np.exp(f1@x)
    SOE2 = lambda x : w2@np.exp(f2@x)
    d = f1.shape[1]

    #X = cartesian_product(*(d*[np.linspace(-10,10,51)]))
    X = cartesian_product(*(d*[np.linspace(-10,10,int(10000**(1/d)))]))
    tmp1 = np.array([SOE1(x) for x in X])
    tmp2 = np.array([SOE2(x) for x in X])
    print(f'err([-{10},{10}]^{d})    \t: ',max(abs(tmp1-tmp2))/max(abs(tmp1)))

# Example 1.1
$$\gamma_1=\begin{pmatrix}3\\2\\1\\2\\1\end{pmatrix}\qquad\text{and}\qquad
\Lambda_1=\begin{pmatrix}
\sqrt{2.21}i&3.33i\\
-5.63i&-\sqrt 5i\\
-3.47i&\sqrt 6 i\\
-\sqrt{7.1}i&-4.5i\\
0.46i&-9.44i
\end{pmatrix}$$

In [20]:
# Setting weights and frequencies
freq = 1j*np.array([[2.21**0.5,3.33],
                    [-5.63,-5**0.5],
                    [-3.47,6**0.5],
                    [-7.1**0.5,-4.5],
                    [0.46,-9.44]])
weig = np.array([3,2,1,2,1])


# Setting Parameters
P    = 4
N    = 15
tau  = 7
Mmax = None


# Creating Array with Fourier Coefficients
coef = FC_arr(weig,freq,N,P)


# Running Code and compare results
print('Results Algorithm 3')
weig1,freq1 = iESPIRA(coef,tau=tau,P = P)
compare(weig,freq,weig1,freq1)
print()

print('Results Algorithm 4')
# Running Code and compare results
weig1,freq1 = rESPIRA(coef,Mmax=Mmax,P = P)
compare(weig,freq,weig1,freq1)

Results Algorithm 3
err(freq)       	:  8.181963402957387e-14
err(weig)       	:  3.212351669189136e-13
err([-10,10]^2)    	:  8.406726045197592e-13

Results Algorithm 4
err(freq)       	:  8.181963402957387e-14
err(weig)       	:  3.010293531911448e-13
err([-10,10]^2)    	:  7.6274899024679e-13


# Example 1.2

$$\gamma_2 = \begin{pmatrix}-1\\-2\\-3\\1\\2\\3\end{pmatrix}\qquad\text{and} \qquad
\Lambda_2=\begin{pmatrix}
-2-3i&\sqrt \pi(-1+i)&0.5i\\
-1+\sqrt{20}i&-3+i&-1+i\\
3i&-4+0.5i&1.22i\\
-2+3i&\sqrt\pi(-1-i)&-0.5i\\
-1-\sqrt{20}i&-3-i&-1-i\\
-3i&-4-0.5i&-1.22i
\end{pmatrix}$$

In [21]:
# Setting weights and frequencies
freq = np.array([[-2-3j,np.pi**0.5*(-1+1j),0.5j],
                    [-1+20**0.5*1j,-3+1j,-1+1j],
                    [3j,-4+0.5j,1.22j],
                    [-2+3j,np.pi**0.5*(-1-1j),-0.5j],
                    [-1-20**0.5*1j,-3-1j,-1-1j],
                    [-3j,-4-0.5j,-1.22j]])
weig = np.array([-1,-2,-3,1,2,3])


# Setting Parameters
P    = 5
N    = 12
tau  = 4
Mmax = None


# Creating Array with Fourier Coefficients
coef = FC_arr(weig,freq,N,P)

# Running Code and compare results
print('Results Algorithm 3')
weig1,freq1 = iESPIRA(coef,tau=tau,P = P)
compare(weig,freq,weig1,freq1)
print()

print('Results Algorithm 4')
# Running Code and compare results
weig1,freq1 = rESPIRA(coef,Mmax=Mmax,P = P)
compare(weig,freq,weig1,freq1)

Results Algorithm 3
err(freq)       	:  5.202512674913886e-10
err(weig)       	:  5.344067111940053e-10
err([-10,10]^3)    	:  1.956598364097439e-09

Results Algorithm 4
err(freq)       	:  1.4014559446253236e-13
err(weig)       	:  8.134421026707991e-14
err([-10,10]^3)    	:  1.6700924150888093e-12


# Example 2.1
$$\gamma_3 =\begin{pmatrix}1\\1\\1\\1\\1\\1\\1\\1\\1\end{pmatrix}\qquad\text{and}\qquad
\Lambda_3 = \begin{pmatrix} 2+2i&0.2i&i&1\\
2+2i&0.2i&i&-1\\
2+2i&-2&1+i&i\\
2+2i&-2&1+i&-2i\\
2+2i&-2&1+i&3i\\
3+i&-\pi&-3&-\sqrt \pi i\\
3+i&-\pi&1&2i\\
3+i&-\pi&1&-4\\
3+i&0.2i&1+i&\sqrt{20}i \end{pmatrix} $$

In [28]:
# Setting weights and frequencies
freq = np.array([[2+2j,0.2j,1j,1],
                 [2+2j,0.2j,1j,-1],
                 [2+2j,-2,1+1j,1j],
                 [2+2j,-2,1+1j,-2j],
                 [2+2j,-2,1+1j,3j],
                 [3+1j,-np.pi,-3,-np.sqrt(np.pi)*1j],
                 [3+1j,-np.pi,1,2j],
                 [3+1j,-np.pi,1,-4],
                 [3+1j,0.2j,1+1j,np.sqrt(20)*1j]])
weig = np.array([ 1,1,1,1,1,1,1,1,1])

P    = 2.4
N    = 10
Mmax = None


# Creating Array with Fourier Coefficients
coef = FC_arr(weig,freq,N,P)

print('Results Algorithm 4')
# Running Code and compare results
weig1,freq1 = rESPIRA(coef,Mmax=Mmax,P = P)
compare(weig,freq,weig1,freq1)

Results Algorithm 4
err(freq)       	:  3.5165355411642337e-13
err(weig)       	:  3.661269806159282e-13
err([-10,10]^4)    	:  4.630010232741065e-13


# Example 2.2


$$\gamma_4 = \begin{pmatrix}1\\-4\\-2\\2\end{pmatrix}\qquad\text{and}\qquad
\Lambda_4 = \begin{pmatrix} 
-1.47-0.27i&-1.87-0.57i&-1.35+4.61i\\
-1.47-0.27i&-1.87-0.57i&-1.26-2.58i\\
-1.47-0.27i&-0.84+7.53i&-1.75-1.33i\\
-0.60+4.86i&-0.13+5.05i&-0.12+8.34i\\
\end{pmatrix}$$

In [29]:
# Setting weights and frequencies
freq = np.array([[-1.47-0.27j, -1.87-0.57j, -1.35+4.61j],
       [-1.47-0.27j, -1.87-0.57j, -1.26-2.58j],
       [-1.47-0.27j, -0.84+7.53j, -1.75-1.33j],
       [-0.6 +4.86j, -0.13+5.05j, -0.12+8.34j]])
weig = np.array([ 1, -4, -2,  2])


# Setting Parameters
P    = 1
N    = 10
Mmax = None


# Creating Array with Fourier Coefficients
coef = FC_arr(weig,freq,N,P)


# Running Code and compare results
weig1,freq1 = rESPIRA(coef,Mmax=Mmax,P = P,LP=0)
compare(weig,freq,weig1,freq1)

err(freq)       	:  3.48657705372779e-15
err(weig)       	:  2.9934237347206168e-15
err([-10,10]^3)    	:  1.5844187106098163e-13


# Example 3.1

$$ \gamma_5 = \begin{pmatrix} 1+i\\2+3i\\5-6i\\0.2-i\\1+i\\2+3i\\5-6i\\0.2-i\end{pmatrix}
\qquad\text{and}\qquad
\Lambda_5 = \begin{pmatrix}
0.1i&1.2i\\
0.19i&1.3i\\
0.3i&1.5i\\
0.35i&0.3i\\
-0.1i&1.2i\\
-0.19i&0.35i\\
-0.3i&-1.5i\\
-0.3i&0.3i
\end{pmatrix}
$$

In [30]:
# Setting weights and frequencies
freq = 1j*np.array([[0.1,1.2],
                    [0.19,1.3],
                    [0.3,1.5],
                    [0.35,0.3],
                    [-0.1,1.2],
                    [-0.19,0.35],
                    [-0.3,-1.5],
                    [-0.3,0.3]])
weig = np.array([1+1j,2+3j,5-6j,0.2-1j,1+1j,2+3j,5-6j,0.2-1j])


# Setting Parameters
P    = 60
N    = 15
Mmax = None



# Creating Array with Fourier Coefficients
coef = FC_arr(weig, freq,N,P)


# Running Code and compare results
weig1,freq1 = rESPIRA(coef,Mmax=Mmax,P = P,prec=1e-13,LP=0,pinfo=0)
compare(weig,freq,weig1,freq1)

err(freq)       	:  1.288066589443214e-14
err(weig)       	:  4.127136147915394e-14
err([-10,10]^2)    	:  3.494974510506943e-14


# Example 3.2

$$ \gamma_6 = \begin{pmatrix} 1+i\\2+3i\\5-6i\\0.2-i\\1+i\\2+3i\\5-6i\\0.2-i\end{pmatrix}
\qquad\text{and}\qquad
\Lambda_6 = \begin{pmatrix}
0.1i&1.2i&0.1i\\
0.19i&1.3i&0.2i\\
0.4i&1.5i&1.5i\\
0.45i&0.3i&-0.3i\\
-0.1i&1.2i&0.1i\\
-0.19i&0.35i&-0.5i\\
-0.4i&-1.5i&0.25i\\
-0.4i&0.3i&-0.3i
\end{pmatrix}
$$

In [31]:
# Setting weights and frequencies
freq = 1j*np.array([[0.1,1.2,0.1],
                    [0.19,1.3,0.2],
                    [0.4,1.5,1.5],
                    [0.45,0.3,-0.3],
                    [-0.1,1.2,0.1],
                    [-0.19,0.35,-0.5],
                    [-0.4,-1.5,0.25],
                    [-0.4,0.3,0.3]])
weig = np.array([1+1j,2+3j,5-6j,0.2-1j,1+1j,2+3j,5-6j,0.2-1j])


# Setting Parameters
P    = 60
N    = 15
Mmax = None



# Creating Array with Fourier Coefficients
coef = FC_arr(weig, freq,N,P)


# Running Code and compare results
weig1,freq1 = rESPIRA(coef,Mmax=Mmax,P = P,prec=1e-13,LP=0,pinfo=0)
compare(weig,freq,weig1,freq1)

err(freq)       	:  2.538663506953771e-15
err(weig)       	:  3.1052113244573774e-14
err([-10,10]^3)    	:  1.7268311228955442e-14


# Example 3.3

$$ \gamma_7 = \begin{pmatrix} 1+i\\2+3i\\5-6i\\0.2-i\\1+i\\2+3i\\5-6i\\0.2-i\end{pmatrix}
\qquad\text{and}\qquad
\Lambda_7 = \begin{pmatrix}
0.1i&1.2i&0.1i&0.45i\\
0.19i&1.3i&0.2i&1.5i\\
0.3i&1.5i&1.5i&-1.3i\\
0.45i&0.3i&-0.3i&0.4i\\
-0.1i&1.2i&0.1i&-1.5i\\
-0.19i&0.35i&-0.5i&-0.45i\\
-0.4i&-1.5i&0.25i&1.3i\\
-0.4i&0.3i&-0.3i&0.4i
\end{pmatrix}
$$

In [32]:
# Setting weights and frequencies
freq = 1j*np.array([[0.1,1.2,0.1,0.45],
                    [0.19,1.3,0.2,1.5],
                    [0.3,1.5,1.5,-1.3],
                    [0.45,0.3,-0.3,0.4],
                    [-0.1,1.2,0.1,-1.5],
                    [-0.19,0.35,-0.5,-0.45],
                    [-0.4,-1.5,0.25,1.3],
                    [-0.4,0.3,-0.3,0.4]])
weig = np.array([1+1j,2+3j,5-6j,0.2-1j,1+1j,2+3j,5-6j,0.2-1j])


# Setting Parameters
P    = 60
N    = 15
Mmax = None



# Creating Array with Fourier Coefficients
coef = FC_arr(weig, freq,N,P)


# Running Code and compare results
weig1,freq1 = rESPIRA(coef,Mmax=Mmax,P = P,prec=1e-13,LP=0,pinfo=0)
compare(weig,freq,weig1,freq1)

err(freq)       	:  1.5535129750025377e-14
err(weig)       	:  6.67037457277657e-14
err([-10,10]^4)    	:  8.251577030745719e-14
